# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zuhairsyed123/ML_internship_Assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Our Answers (Data Contract):**
1. **Unit of Analysis**: The unit of analysis is a single content page (`content_hash_id`) for a specific client (`client_hash_id`) on a given day (`report_date`).
2. **Table(s) used**: `dim_content` (metadata) joined with `fact_content_daily_performance` (daily search & traffic metrics).
3. **Time Window**: We use a mid-panel month, specifically March 2026 (`month=2026-03`), as our feature window. This avoids peeking into the final test month (June 2026). Our target performance outcome is evaluated in the next month, April 2026 (`month=2026-04`).
4. **Label or Proxy**: We predict performance decline (proxy target: next month's GSC impressions drop by >20% compared to the current month's GSC impressions).
5. **Deliberate Exclusion**: We exclude the `trend_direction` and `trend_pct` columns from our feature set because they are derived from the target time window and leak the label.

In [1]:
# Setup environment and authenticate
import os
import duckdb
import pandas as pd

# Load Hugging Face token from .env
env_path = "../../.env"
token = None
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            if line.startswith("HF_TOKEN="):
                token = line.split("=", 1)[1].strip()
                break

if not token:
    raise ValueError("HF_TOKEN not found in .env")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
print("DuckDB connected and authenticated successfully.")

DuckDB connected and authenticated successfully.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Our Classification:**
- **Feature** (Available at the decision moment):
  - `gsc_impressions` (historical search impressions) - knowable before the decision point as it reflects historical search demand.
  - `gsc_clicks` (historical click volume) - knowable before the decision point as it reflects historical search engagement.
  - `gsc_avg_position` (historical average position) - knowable before the decision point as it reflects historical ranking.
  - `ga4_sessions` (historical session count) - knowable before the decision point as it reflects historical site-level traffic.
  - `word_count` (static content length from `dim_content`) - knowable before the decision point as it describes the published page's structural length.
- **Label / Proxy** (The outcome we predict):
  - `is_declining` (derived target) - indicating whether next-month impressions drop by more than 20% compared to current-month impressions.
- **Context** (Used only for grouping, splitting, and joining, never fed directly to the model):
  - `client_hash_id`, `content_hash_id`, `report_date`
- **Excluded** (Private, future, or leaking information):
  - `trend_direction` and `trend_pct` - excluded because they are computed over the prediction window and leak the performance decline target.
  - Future performance metrics (like future clicks/impressions) - excluded from features because they are from the target window and would cause absolute feature leakage.

In [2]:
# Print the columns we plan to use from dim_content and fact_content_daily_performance
print("Planned features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, word_count")
print("Planned target: is_declining (derived from future gsc_impressions)")
print("Planned context: client_hash_id, content_hash_id, report_date")

Planned features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, word_count
Planned target: is_declining (derived from future gsc_impressions)
Planned context: client_hash_id, content_hash_id, report_date


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We will run 3 verification queries on a mid-panel month (`month=2026-03`):
1. **Query 1 (Grain Check)**: Verify that grouping by `report_date + client_hash_id + content_hash_id` has no duplicate rows (count > 1 is 0).
2. **Query 2 (Counts & Date Span)**: Query the total row count, minimum date, and maximum date in the partition.
3. **Query 3 (Availability Check)**: Count the total rows in the partition and show how many survive when filtering with `ga4_data_available IS TRUE`.

Then, we will extract a 5-feature frame and run a **Leakage Trap Experiment** comparing an Honest model with a Leaked model (which includes future clicks `clk_future` as a feature).

In [3]:
print("--- Fact 1: Grain Check ---")
grain_query = f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
"""
grain_df = con.sql(grain_query).df()
print(f"Duplicate rows matching grain: {len(grain_df)} (Expected: 0)")

print("\n--- Fact 2: Row Count & Date Span ---")
counts_query = f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print(con.sql(counts_query).df())

print("\n--- Fact 3: Availability Check ---")
avail_query = f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(CASE WHEN ga4_data_available = TRUE THEN 1 END) AS ga4_available_rows,
           COUNT(CASE WHEN ga4_data_available = TRUE THEN 1 END) * 1.0 / COUNT(*) AS availability_share
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print(con.sql(avail_query).df())

print("\n--- Extracting Features for Leakage Experiment ---")
feature_query = f"""
    WITH current_month AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_current,
               SUM(gsc_clicks) AS clk_current,
               AVG(gsc_avg_position) AS pos_current,
               SUM(ga4_sessions) AS ses_current
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    next_month AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_future,
               SUM(gsc_clicks) AS clk_future
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT c.client_hash_id, c.content_hash_id,
           c.imp_current, c.clk_current, c.pos_current, c.ses_current,
           coalesce(n.imp_future, 0) AS imp_future,
           coalesce(n.clk_future, 0) AS clk_future,
           dc.word_count
    FROM current_month c
    LEFT JOIN next_month n ON c.client_hash_id = n.client_hash_id AND c.content_hash_id = n.content_hash_id
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') dc ON c.content_hash_id = dc.content_hash_id
    WHERE c.imp_current >= 100
"""
df = con.sql(feature_query).df()
print(f"Extracted {len(df):,} content items with current monthly volume >= 100 impressions.")

# Fill missing values safely
df['pos_current'] = df['pos_current'].fillna(0)
df['ses_current'] = df['ses_current'].fillna(0)
df['word_count'] = df['word_count'].fillna(0)

# Define Target: Decline > 20% in future month impressions
df['is_declining'] = (df['imp_future'] < 0.8 * df['imp_current']).astype(int)

# Define Feature Sets
features_honest = ['imp_current', 'clk_current', 'pos_current', 'ses_current', 'word_count']
features_leaked = features_honest + ['imp_future']  # Future impressions directly leak the outcome

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score

X_honest = df[features_honest]
X_leaked = df[features_leaked]
y = df['is_declining']

X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.3, random_state=42, stratify=y)

# Train Honest Model
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_h, y_tr)
y_pred_h = model_honest.predict(X_te_h)
score_h = accuracy_score(y_te, y_pred_h)
prec_h = precision_score(y_te, y_pred_h)

# Train Leaked Model
model_leaked = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr)
y_pred_l = model_leaked.predict(X_te_l)
score_l = accuracy_score(y_te, y_pred_l)
prec_l = precision_score(y_te, y_pred_l)

print("\n--- Leakage Trap Experiment Results ---")
print(f"Honest Model Accuracy:  {score_h:.4f}")
print(f"Honest Model Precision: {prec_h:.4f}")
print(f"Leaked Model Accuracy:  {score_l:.4f} (Near Perfect!)")
print(f"Leaked Model Precision: {prec_l:.4f} (Near Perfect!)")

importances = pd.Series(model_leaked.feature_importances_, index=features_leaked).sort_values(ascending=False)
print("\nLeaked Model Feature Importances (Notice how imp_future dominates!):")
print(importances)

--- Fact 1: Grain Check ---


Duplicate rows matching grain: 0 (Expected: 0)

--- Fact 2: Row Count & Date Span ---


   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

--- Fact 3: Availability Check ---


   total_rows  ga4_available_rows  availability_share
0     9841378              413966            0.042064

--- Extracting Features for Leakage Experiment ---


Extracted 101,441 content items with current monthly volume >= 100 impressions.



--- Leakage Trap Experiment Results ---
Honest Model Accuracy:  0.6154
Honest Model Precision: 0.6301
Leaked Model Accuracy:  0.9866 (Near Perfect!)
Leaked Model Precision: 0.9883 (Near Perfect!)

Leaked Model Feature Importances (Notice how imp_future dominates!):
imp_future     0.481798
imp_current    0.365868
word_count     0.048822
pos_current    0.045220
clk_current    0.031519
ses_current    0.026772
dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Key Limitations identified in this data:**
1. **Unbalanced History Depth**: Different clients have histories starting at different times (`gsc_data_start`, `ga4_data_start`). A global calendar-based time window will result in missing or misleading data for clients with shorter history. We must use relative client-specific windows instead.
2. **GA4 Early Missingness**: For rows before a client's `ga4_data_start`, GA4 metrics are zero-filled with `ga4_data_available = FALSE`. Treating these as genuine zero traffic/engagement is a bug; they must be filtered out using `ga4_data_available IS TRUE`.
3. **Window Overlap & Leakage**: If features are calculated over trailing windows that overlap with the future label window, information will leak. We must keep our feature windows and target windows strictly separated.
4. **Causal Attribution**: This is purely observational data. We can prioritize which pages are good review candidates, but we cannot prove that a refresh will cause a recovery without running an A/B test or using causal inference methods.

In [4]:
# Query dim_clients to show the unbalanced start dates for client data
clients_df = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM read_parquet('{REL}/dim_clients.parquet')
    ORDER BY gsc_data_start NULLS LAST
    LIMIT 10
""").df()
print("Unbalanced client history start dates:")
print(clients_df)

Unbalanced client history start dates:
            client_hash_id gsc_data_start ga4_data_start
0  client_9958f0a7ae1df715     2025-01-27     2025-10-29
1  client_ff644d8251367cbb     2025-01-27     2025-10-29
2  client_73cda7b4e4f265ea     2025-02-11     2026-03-24
3  client_fef1a8f436438636     2025-03-11     2026-03-06
4  client_62f4a7e64f5e0096     2025-06-07            NaT
5  client_b10cb2997d0c7c86     2025-06-18     2025-11-15
6  client_65de48885f4ef01b     2025-06-21     2026-02-19
7  client_c182d11e4862a37d     2025-06-21     2026-02-20
8  client_3197e6291363b4db     2025-06-29     2025-11-09
9  client_625b6439094e23e4     2025-07-01     2026-02-19


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.